**Cell 1**

# 01 — Data Acquisition & Inventory

We pull each class as its **own pre-annotated dataset** from Roboflow / Kaggle (YOLO format). One folder per class:

```
data/sources/
├── projector/
├── whiteboard/
├── fire_extinguisher/
└── door_sign/
```

Each source folder is a full Roboflow export (images + YOLO `.txt` labels, possibly with a `data.yaml`). Because each download is a **single-class dataset**, every label inside has `class_id = 0`. Notebook **02** handles the global remap (0 → 0/1/2/3).

This notebook just **inventories** what was downloaded and sanity-checks that every image has a matching label.

In [ ]:
# Cell 2 — install dependencies for this notebook
%pip install -q pandas pillow pyyaml

In [ ]:
# Cell 3 — setup
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image

CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
SRC = Path('../data/sources')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
assert SRC.exists(), f'put per-class Roboflow/Kaggle exports under {SRC.resolve()}'

**Cell 4**

## 1.1 How to populate `data/sources/`

For each class, download the dataset as a **YOLOv8 / YOLOv11** export (same label format) and unzip into the matching folder. A Roboflow export usually looks like:

```
data/sources/projector/
├── train/{images,labels}
├── valid/{images,labels}
├── test/{images,labels}
└── data.yaml
```

Kaggle datasets may be flatter (e.g. `images/` + `labels/` at the root). Both layouts are fine — the recursive walks below discover all image/label pairs regardless of nesting.

**Cell 5**

## 1.2 Inventory every class folder
Count images, count labels, and check that each image has a sibling `.txt` label under the same parent.

In [ ]:
# Cell 6 — walk each class folder
def find_label(img_path: Path) -> Path | None:
    # Roboflow/Kaggle convention: sibling 'labels/<stem>.txt' to images/ dir
    candidates = [
        img_path.parent.parent / 'labels' / (img_path.stem + '.txt'),
        img_path.with_suffix('.txt'),
    ]
    return next((c for c in candidates if c.exists()), None)

rows = []
for c in CLASSES:
    folder = SRC / c
    if not folder.exists():
        print(f'[missing] {folder}')
        continue
    for img in folder.rglob('*'):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        lbl = find_label(img)
        rows.append({
            'class': c,
            'image_path': str(img.relative_to(SRC)),
            'label_path': str(lbl.relative_to(SRC)) if lbl else None,
            'has_label': lbl is not None,
        })

inv = pd.DataFrame(rows)
inv.groupby('class').agg(images=('image_path', 'count'),
                         with_label=('has_label', 'sum'))

**Cell 7**

## 1.3 Read each dataset's `data.yaml` (Roboflow) if present
Useful to confirm the source dataset's class names (should be a single class per folder).

In [ ]:
# Cell 8 — peek at source data.yaml files
for c in CLASSES:
    y = list((SRC / c).rglob('data.yaml'))
    if not y:
        print(f'{c}: no data.yaml (likely a Kaggle-flat export)')
        continue
    cfg = yaml.safe_load(y[0].read_text())
    print(f'{c}: names={cfg.get("names")}  nc={cfg.get("nc")}  file={y[0]}')

**Cell 9**

## 1.4 Confirm label format is YOLO and class_id is 0
Each source dataset is single-class, so every label row should start with `0`. If a dataset starts with something else, note it — notebook 02 will remap explicitly.

In [ ]:
# Cell 10 — tally the class ids that appear inside each source
from collections import Counter
summary = []
for c in CLASSES:
    counter = Counter()
    for lbl in (SRC / c).rglob('*.txt'):
        if lbl.name == 'classes.txt':
            continue
        for line in lbl.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                counter[parts[0]] += 1
    summary.append({'class': c, **counter})
pd.DataFrame(summary).fillna(0)

**Cell 11**

## 1.5 Target volume per class

| class             | target images |
|-------------------|---------------|
| projector         | ~200 |
| whiteboard        | ~200 |
| fire_extinguisher | ~200 |
| door_sign         | ~200 |

If a source has more than 200 images, notebook 02 will downsample per class; if fewer, pull from another Roboflow/Kaggle project and re-run this inventory.